In [1]:
import pandas as pd
import numpy as np
import re
import string

og_paris_file = pd.read_csv('/Users/chris/Rag bot learning/building-llm-applications-from-scratch/Module 4/Semantic_Search/paris_02_11_23.csv', encoding='utf-8')

og_paris_file.head()






,review_id,date,review_rating,title,text,votes,url,language,platform,author_id,author_name,author_username,name,id,description,rating,rating_count,features
0,864290614,2022-10-12,1,A large impersonal place with an on time check...,"If you are looking for a huge, grand hotel exp...",1,/ShowUserReviews-g187147-d207742-r864290614-In...,en,MOBILE,E488EBBA1F82F16BF878FE274C735941,Anna J,AnnaJ250,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature..."
1,864049819,2022-10-10,4,Good hotel with rude waiter,We went to this hotel just this month\nWe have...,1,/ShowUserReviews-g187147-d207742-r864049819-In...,en,MOBILE,4A830AD8B128F60AC02E83D6B6A530F7,QATAR2007,QATAR2007,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature..."
2,863952022,2022-10-10,5,Fantastic,"Absolutely top-notch. Room, service, bed, pill...",0,/ShowUserReviews-g187147-d207742-r863952022-In...,en,OTHER,AA2958C3E083861E81EEC085671BAA5B,aji1376,aji1376,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature..."
3,863793066,2022-10-09,4,"Amidst the chaos of Fashion week, their servic...",We stayed during the Paris Fashion Week Chaos....,0,/ShowUserReviews-g187147-d207742-r863793066-In...,en,MOBILE,DE4AB96DA3E104846D6D6423C2DAA4C8,jelinc2016,jelinc2016,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature..."
4,863631994,2022-10-08,2,Not worth the effort or money,This hotel is not worth the effort or the pric...,0,/ShowUserReviews-g187147-d207742-r863631994-In...,en,MOBILE,DE02D713F209AEC684DC6108509E6912,VikaasK,VikaasK,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature..."


In [ ]:
og_paris_file ["combined"] = (
    "title: " + og_paris_file.title.str.strip()+"; text: " + og_paris_file.text.str.strip()
    # +"; desc: "+ og_paris_file.text.str.strip()
)

paris_file = og_paris_file.copy()

print(paris_file.to_string())





In [9]:
paris_file.head()

,review_id,date,review_rating,title,text,votes,url,language,platform,author_id,author_name,author_username,name,id,description,rating,rating_count,features,combined
0,864290614,2022-10-12,1,A large impersonal place with an on time check...,"If you are looking for a huge, grand hotel exp...",1,/ShowUserReviews-g187147-d207742-r864290614-In...,en,MOBILE,E488EBBA1F82F16BF878FE274C735941,Anna J,AnnaJ250,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature...",title: A large impersonal place with an on tim...
1,864049819,2022-10-10,4,Good hotel with rude waiter,We went to this hotel just this month\nWe have...,1,/ShowUserReviews-g187147-d207742-r864049819-In...,en,MOBILE,4A830AD8B128F60AC02E83D6B6A530F7,QATAR2007,QATAR2007,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature...",title: Good hotel with rude waiter; text: We w...
2,863952022,2022-10-10,5,Fantastic,"Absolutely top-notch. Room, service, bed, pill...",0,/ShowUserReviews-g187147-d207742-r863952022-In...,en,OTHER,AA2958C3E083861E81EEC085671BAA5B,aji1376,aji1376,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature...",title: Fantastic; text: Absolutely top-notch. ...
3,863793066,2022-10-09,4,"Amidst the chaos of Fashion week, their servic...",We stayed during the Paris Fashion Week Chaos....,0,/ShowUserReviews-g187147-d207742-r863793066-In...,en,MOBILE,DE4AB96DA3E104846D6D6423C2DAA4C8,jelinc2016,jelinc2016,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature...","title: Amidst the chaos of Fashion week, their..."
4,863631994,2022-10-08,2,Not worth the effort or money,This hotel is not worth the effort or the pric...,0,/ShowUserReviews-g187147-d207742-r863631994-In...,en,MOBILE,DE02D713F209AEC684DC6108509E6912,VikaasK,VikaasK,InterContinental Paris - Le Grand,207742,"The InterContinental Paris Le Grand, opened du...",4.5,3517.0,"['roomFeatures_air conditioning', 'roomFeature...",title: Not worth the effort or money; text: Th...


In [ ]:

clean_paris_file = paris_file.sort_values(by='name', ascending=True).groupby('name', sort=False).agg({
    'text': ' '.join,
    'review_rating': ['min','max','mean'],
    'date': 'first',
    'votes': 'sum',
    'description': 'first',
}).reset_index()


In [22]:
clean_paris_file.head()

name                                               text  \
                                                                 join   
0         1K Paris  Stayed here for 3 nights between 12 - 15 Janua...   
1   3 Ducks Hostel  This was my least favorite hostel in my travel...   
2    9Confidentiel  This hotel is in such a great location for vis...   
3  A Room In Paris  beautiful bldg; wonderful breakfast; clean, co...   
4    Amastan Paris  Was charged €50 extra after chk out on a basel...   

  review_rating                    date votes  \
            min max    mean       first   sum   
0             1   5  4.5875  2021-01-17    61   
1             1   5  3.8250  2018-10-24    34   
2             1   5  4.5375  2019-10-12    30   
3             1   5  4.6375  2017-05-11   170   
4             1   5  4.5125  2017-05-09    45   

                                         description  
                                               first  
0                                               None  
1  Authentique,Design, confortable, idealy locate...  
2                                               None  
3  Right in the city centre of Paris 5 beautiful ...  
4                                               None

In [20]:
paris_newer = pd.read_csv('/Users/chris/Rag bot learning/building-llm-applications-from-scratch/Module 4/streamlit/paris_clean_newer.csv')
paris_newer.head()

,id,date,rating,title,text,votes,url,language,platform,author_id,author_name,author_username,Hotel,description,price_per_night
0,861079521,2022-09-20,5,Loved our stay at Hotel Malte,Choosing a hotel in Paris is always a difficul...,0,/ShowUserReviews-g187147-d228694-r861079521-Ho...,en,OTHER,92EE2AAD926B93A7188923E86E2DF52A,PeggyArchambault,PeggyArchambault,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,$395
1,851712783,2022-08-03,5,Number one in Paris,The reception was friendly and professional an...,0,/ShowUserReviews-g187147-d228694-r851712783-Ho...,en,MOBILE,844015936EC232C4EA6265B76AA5C95A,MikeUys,MikeUys,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,$395
2,861070340,2022-09-20,5,Wonderful Stay All Around,"Wonderful room and location, as well as kind, ...",0,/ShowUserReviews-g187147-d228694-r861070340-Ho...,en,OTHER,78852954925A4EBC0168D0B323385766,ssharie101,ssharie101,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,$395
3,861023809,2022-09-20,5,Hotel Malte,We spent one night here on our way south. The ...,0,/ShowUserReviews-g187147-d228694-r861023809-Ho...,en,OTHER,1F6A616919DA5F736B3966A68029F3C2,Gareth J,164garethj,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,$395
4,860944034,2022-09-19,5,Fabulous short stay,Walking into hotel for first time we were impr...,0,/ShowUserReviews-g187147-d228694-r860944034-Ho...,en,MOBILE,F5C46BDFFA45D0FD1B2EA09FEC976D74,bobbyboo,BlueTanzanite,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,$395


In [21]:
paris_newer.shape

(7321, 15)